In [1]:
import sys
from pathlib import Path

for base in (Path.cwd(), Path.cwd().parent):
    src = base / "src"
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))


# Damage Predictor
Train, save, load, and inspect the corruption-vector damage model.

In [2]:
import numpy as np
import pandas as pd
from driftalytics import DamagePredictor, compute_corruption_vector
from driftalytics.train_damage import train_and_save

model_path = "models/damage_predictor.json"
model = train_and_save(model_path, n=1000, seed=3)
print("model metrics:", model.metrics_)

trained damage predictor: rows=1000 train=800 test=200 MAE=0.0248 RMSE=0.0311 R2=0.9740
saved to models/damage_predictor.json
model metrics: {'mae': 0.025130126404008784, 'rmse': 0.03187724407070463, 'r2': 0.975765159355896}


In [3]:
rng = np.random.default_rng(0)
reference = pd.DataFrame({"a": rng.normal(size=500), "b": rng.normal(size=500)})
live = pd.DataFrame({"a": rng.normal(loc=1.0, size=200), "b": rng.normal(size=200)})

vector = compute_corruption_vector(reference, live, expected_size=len(live))
print("corruption vector:", vector.as_dict())
print("estimated performance drop:", round(model.predict(vector), 4))

corruption vector: {'drift_magnitude': 0.6009847718777183, 'missingness_delta': 0.0, 'cardinality_shift': 0.21231082391825762, 'schema_change': 0.0, 'volume_delta': 0.0}
estimated performance drop: 0.2564


In [4]:
loaded = DamagePredictor.load(model_path)
print("loaded prediction:", round(loaded.predict(vector), 4))
print("matches saved model:", loaded.predict(vector) == model.predict(vector))

loaded prediction: 0.2564
matches saved model: True
